# Qwen2.5 3B Trace Explainer Final Adapter

Purpose: train the model to rewrite locked calculator traces into concise English explanations. The input already contains the final answer, unit, formula, and calculation from the calculator. The model must only explain that locked trace and must not change the answer, unit, formula, or calculation.

Dataset expected on Kaggle: `trace_explainer_final_train.jsonl` and `trace_explainer_final_valid.jsonl`. Seed files are intentionally ignored.


In [ ]:
!pip -q install -U "transformers>=4.46" "datasets>=2.20" "peft>=0.12" "accelerate>=0.33" "bitsandbytes>=0.46.1" safetensors


In [ ]:
import json, os, time
from pathlib import Path

ADAPTER_KEY = 'trace_explainer_final'
ADAPTER_NAME = 'qwen25_3b_trace_explainer_final_adapter'
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
OUTPUT_DIR = Path('/kaggle/working') / ADAPTER_NAME
MAX_SEQ_LENGTH = 1536
EPOCHS = 3
LR = 1.5e-4
REQUIRED_KEYS = {'explanation'}
FORBIDDEN_KEYS = {'answer', 'unit', 'python_code', 'final_result'}
print('Output:', OUTPUT_DIR)


In [ ]:
def find_jsonl(split):
    roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]
    wanted = {f'{ADAPTER_KEY}_{split}.jsonl'}
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob('*.jsonl'):
            low = str(path).lower()
            if path.name in wanted and ADAPTER_KEY in low and '.seed.' not in path.name:
                candidates.append(path)
    if not candidates:
        raise FileNotFoundError(
            f'Could not find {ADAPTER_KEY}_{split}.jsonl. Upload the full trace_explainer_final folder, not only seed files.'
        )
    return sorted(candidates, key=lambda p: (len(str(p)), str(p)))[0]

def count_jsonl(path):
    return sum(1 for line in open(path, encoding='utf-8') if line.strip())

TRAIN_FILE = find_jsonl('train')
VALID_FILE = find_jsonl('valid')
print('Train:', TRAIN_FILE, 'rows:', count_jsonl(TRAIN_FILE))
print('Valid:', VALID_FILE, 'rows:', count_jsonl(VALID_FILE))
summary_candidates = list(TRAIN_FILE.parent.glob('*summary.json'))
if summary_candidates:
    print(json.dumps(json.load(open(summary_candidates[0], encoding='utf-8')), indent=2, ensure_ascii=False)[:1800])


In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def validate_rows(rows, split):
    bad = []
    for row in rows:
        messages = row.get('messages') or []
        if len(messages) < 3:
            bad.append((row.get('id'), 'messages_too_short'))
            continue
        try:
            obj = json.loads(messages[-1]['content'])
        except Exception as exc:
            bad.append((row.get('id'), f'invalid_json:{exc}'))
            continue
        missing = sorted(REQUIRED_KEYS - set(obj))
        forbidden = sorted(FORBIDDEN_KEYS & set(obj))
        if missing or forbidden:
            bad.append((row.get('id'), f'missing={missing}; forbidden={forbidden}'))
    if bad:
        print(split, 'bad rows:', bad[:10])
        raise ValueError(f'{split} validation failed: {len(bad)} bad rows')
    print(split, 'validated rows:', len(rows))

train_rows = load_jsonl(TRAIN_FILE)
valid_rows = load_jsonl(VALID_FILE)
validate_rows(train_rows, 'train')
validate_rows(valid_rows, 'valid')


In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def render_text(row):
    return tokenizer.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=False)

# Render text before creating the Dataset: metadata contains mixed scalar types
# such as int and empty string, which can break PyArrow inference.
train_texts = [render_text(row) for row in train_rows]
valid_texts = [render_text(row) for row in valid_rows]
train_ds = Dataset.from_dict({'text': train_texts})
valid_ds = Dataset.from_dict({'text': valid_texts})
print('Rendered train rows:', len(train_ds))
print('Rendered valid rows:', len(valid_ds))
print(train_ds[0]['text'][:1000])

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
valid_tok = valid_ds.map(tokenize, batched=True, remove_columns=valid_ds.column_names)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [ ]:
from transformers import TrainingArguments, Trainer

common_args = dict(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=LR,
    warmup_ratio=0.03,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    save_total_limit=2,
    fp16=True,
    optim='paged_adamw_8bit',
    report_to='none',
)
try:
    args = TrainingArguments(eval_strategy='steps', **common_args)
except TypeError:
    args = TrainingArguments(evaluation_strategy='steps', **common_args)

trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=valid_tok, data_collator=collator)
trainer.train()

adapter_dir = OUTPUT_DIR / 'adapter'
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
metadata = {
    'base_model': BASE_MODEL,
    'adapter_purpose': ADAPTER_KEY,
    'train_rows': len(train_rows),
    'valid_rows': len(valid_rows),
    'max_seq_length': MAX_SEQ_LENGTH,
    'epochs': EPOCHS,
    'learning_rate': LR,
    'required_output_keys': sorted(REQUIRED_KEYS),
    'forbidden_output_keys': sorted(FORBIDDEN_KEYS),
    'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
}
(adapter_dir / 'training_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved adapter to', adapter_dir)
print(json.dumps(metadata, indent=2))
